# CIFAR-10 Hyperparameter Search

CIFAR-10 classification hyperparameter search for the perturbation-learning experiments.

The notebook has three stages: frozen-backprop sigma diagnostics, the documented local 3x3 training grid, and an editable full-length run for selected hyperparameters. The model and learning-rule updates are imported from `learning_rules_MLP.py` through the shared grid-search utilities.


In [ ]:
from pathlib import Path
import os
import sys
import tempfile


def is_project_root(path: Path) -> bool:
    return (
        (path / "learning_rules_MLP.py").is_file()
        and (path / "grid-search" / "grid_search_utils.py").is_file()
        and (path / "final-config-runs" / "final_config_utils.py").is_file()
    )


def project_root_candidates():
    starts = [Path.cwd()]
    for env_name in ["PROJECT_ROOT", "COLAB_PROJECT_ROOT"]:
        value = os.environ.get(env_name)
        if value:
            starts.append(Path(value))
    starts.extend([
        Path("/content/backprop-alternatives"),
        Path("/content/drive/MyDrive/backprop-alternatives"),
        Path("/content/drive/MyDrive/colab-folder"),
    ])

    seen = set()
    for start in starts:
        try:
            resolved = start.expanduser().resolve()
        except Exception:
            continue
        for candidate in [resolved, *resolved.parents]:
            if candidate not in seen:
                seen.add(candidate)
                yield candidate


def find_project_root(search_drive: bool = False) -> Path | None:
    for candidate in project_root_candidates():
        if is_project_root(candidate):
            return candidate

    if search_drive:
        for root in [Path("/content"), Path("/content/drive/MyDrive")]:
            if not root.exists():
                continue
            for hit in root.rglob("learning_rules_MLP.py"):
                candidate = hit.parent
                if is_project_root(candidate):
                    return candidate
    return None


PROJECT_ROOT = find_project_root(search_drive=False)
if PROJECT_ROOT is None:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass
    PROJECT_ROOT = find_project_root(search_drive=True)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root. Expected learning_rules_MLP.py, "
        "grid-search/grid_search_utils.py, and final-config-runs/final_config_utils.py."
    )

GRID_SEARCH_DIR = PROJECT_ROOT / "grid-search"
FINAL_CONFIG_DIR = PROJECT_ROOT / "final-config-runs"
DATA_DIR = PROJECT_ROOT / "data"

os.chdir(GRID_SEARCH_DIR)
for path in [PROJECT_ROOT, GRID_SEARCH_DIR, FINAL_CONFIG_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

mpl_cache = Path(tempfile.gettempdir()) / "grid_search_matplotlib_cache"
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

from IPython.display import display

from grid_search_utils import (
    archive_grid_search_outputs,
    build_grid,
    run_full_sweep,
    run_local_grid_search,
    run_sigma_search,
    save_config_snapshot,
    top_grid_rows,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {Path.cwd()}")


## Task Configuration

The local grids match Table 4 in the thesis methods chapter. For the sigma-search stage, each perturbation method uses the three documented sigma values plus one additional value on each side.


In [ ]:
TASK_CONFIG = {
    "task_key": "cifar10",
    "display_name": "CIFAR-10",
    "task_type": "classification",
    "data_loader": "load_cifar10",
    "activation": "relu",
    "output_dir": "grid_search_outputs/cifar10",
    "dimensions": (32 * 32 * 3, 512, 256, 10),
    "methods": ["bp", "np", "np_fixed", "np_fan_in", "wp"],
    "data_kwargs": {
        "train_limit": None,
        "test_limit": None,
        "train_eval_limit": 4000,
        "batch_size": 128,
        "eval_batch_size": 1024,
        "data_dir": str(DATA_DIR / "torchvision"),
        "seed": 0,
        "mean_center_only": True,
        "num_workers": 0,
    },
    "local_grids": {
        "bp": build_grid({"lr": [0.075, 0.100, 0.125]}),
        "np": build_grid({"lr": [0.0100, 0.0120, 0.0140], "sigma": [0.016, 0.0175, 0.01875]}),
        "np_fan_in": build_grid({"lr": [0.00550, 0.00667, 0.00800], "sigma": [0.0050, 0.0055, 0.0060]}),
        "np_fixed": build_grid({"lr": [0.00320, 0.00400, 0.00480], "sigma": [0.165, 0.190, 0.215]}),
        "wp": build_grid({"lr": [0.00120, 0.00160, 0.00200], "sigma": [0.0110, 0.0125, 0.0140]}),
    },
    "sigma_search": {
        "epochs": 50,
        "bp_lr": 0.1,
        "num_perturbations": 10,
        "batch_size": 128,
        "checkpoint_epochs": [1, 25, 50],
        "seeds": [0],
        "sigma_grids": {
            "np": [0.0145, 0.016, 0.0175, 0.01875, 0.0200],
            "np_fan_in": [0.0045, 0.0050, 0.0055, 0.0060, 0.0065],
            "np_fixed": [0.140, 0.165, 0.190, 0.215, 0.240],
            "wp": [0.0095, 0.0110, 0.0125, 0.0140, 0.0155],
        },
    },
    "grid_search": {
        "epochs": 8,
        "seeds": [0],
        "methods": ["bp", "np", "np_fan_in", "np_fixed", "wp"],
        "print_every_epoch": 2,
    },
    "full_run": {
        "epochs": 80,
        "seeds": [0],
        "print_every_epoch": 10,
    },
}

save_config_snapshot(TASK_CONFIG, project_root=PROJECT_ROOT)
TASK_CONFIG


## 1. Frozen-Backprop Sigma Search

This stage trains backpropagation to frozen checkpoints, then evaluates each perturbation method over five candidate sigma values without applying parameter updates. The diagnostics are cosine similarity and coordinate-averaged estimator variance.


In [ ]:
sigma_outputs = run_sigma_search(TASK_CONFIG, project_root=PROJECT_ROOT)

display(sigma_outputs["overall_summary_df"])


## 2. Local 3x3 Training Grid

This stage runs the local grid reported in the thesis. For perturbation methods, the grid is the Cartesian product of three learning rates and three sigma values. Backpropagation is swept over learning rate only.


In [ ]:
grid_outputs = run_local_grid_search(TASK_CONFIG, project_root=PROJECT_ROOT)

display(top_grid_rows(grid_outputs["grid_summary_df"], top_k=5))
display(grid_outputs["best_df"])


## 3. Full-Length Run for Selected Hyperparameters

Edit `FULL_RUN_CONFIGS` if you want to test another hyperparameter setting. By default, these are the selected settings reported in the thesis table.


In [ ]:
FULL_RUN_CONFIGS = {
    "bp": {"lr": 0.100},
    "np": {"lr": 0.0120, "sigma": 0.0175},
    "np_fan_in": {"lr": 0.00667, "sigma": 0.0055},
    "np_fixed": {"lr": 0.00400, "sigma": 0.190},
    "wp": {"lr": 0.00160, "sigma": 0.0125},
}

FULL_RUN_CONFIGS


In [ ]:
full_outputs = run_full_sweep(TASK_CONFIG, FULL_RUN_CONFIGS, project_root=PROJECT_ROOT)

display(full_outputs["run_summary_df"])


## Archive Outputs

This cell zips the generated CSV files and tables for convenient download or upload.


In [ ]:
archive_path = archive_grid_search_outputs(TASK_CONFIG, project_root=PROJECT_ROOT)
archive_path

# In Colab, uncomment to download directly.
# from google.colab import files
# files.download(str(archive_path))
